In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_supplier, load_orders, load_customer, load_nation, q07
DATA_ROOT = "/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)



In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)



In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 3 ###


def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 4 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 5 ###



lineitem_filtered = lineitem[
    (lineitem["L_SHIPDATE"] >= pd.Timestamp("1995-01-01"))
    & (lineitem["L_SHIPDATE"] < pd.Timestamp("1997-01-01"))
]
lineitem_filtered["L_YEAR"] = lineitem_filtered["L_SHIPDATE"].dt.year
lineitem_filtered["VOLUME"] = lineitem_filtered["L_EXTENDEDPRICE"] * (
    1.0 - lineitem_filtered["L_DISCOUNT"]
)
lineitem_filtered = lineitem_filtered.loc[
    :, ["L_ORDERKEY", "L_SUPPKEY", "L_YEAR", "VOLUME"]
]
supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NATIONKEY"]]
orders_filtered = orders.loc[:, ["O_ORDERKEY", "O_CUSTKEY"]]
customer_filtered = customer.loc[:, ["C_CUSTKEY", "C_NATIONKEY"]]
n1 = nation[(nation["N_NAME"] == "FRANCE")].loc[:, ["N_NATIONKEY", "N_NAME"]]
n2 = nation[(nation["N_NAME"] == "GERMANY")].loc[:, ["N_NATIONKEY", "N_NAME"]]

# ----- do nation 1 -----
N1_C = customer_filtered.merge(
    n1, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N1_C = N1_C.drop(columns=["C_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "CUST_NATION"}
)
N1_C_O = N1_C.merge(
    orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="inner"
)
N1_C_O = N1_C_O.drop(columns=["C_CUSTKEY", "O_CUSTKEY"])

N2_S = supplier_filtered.merge(
    n2, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N2_S = N2_S.drop(columns=["S_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "SUPP_NATION"}
)
N2_S_L = N2_S.merge(
    lineitem_filtered, left_on="S_SUPPKEY", right_on="L_SUPPKEY", how="inner"
)
N2_S_L = N2_S_L.drop(columns=["S_SUPPKEY", "L_SUPPKEY"])

total1 = N1_C_O.merge(
    N2_S_L, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner"
)
total1 = total1.drop(columns=["O_ORDERKEY", "L_ORDERKEY"])

# ----- do nation 2 -----
N2_C = customer_filtered.merge(
    n2, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N2_C = N2_C.drop(columns=["C_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "CUST_NATION"}
)
N2_C_O = N2_C.merge(
    orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="inner"
)
N2_C_O = N2_C_O.drop(columns=["C_CUSTKEY", "O_CUSTKEY"])

N1_S = supplier_filtered.merge(
    n1, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
N1_S = N1_S.drop(columns=["S_NATIONKEY", "N_NATIONKEY"]).rename(
    columns={"N_NAME": "SUPP_NATION"}
)
N1_S_L = N1_S.merge(
    lineitem_filtered, left_on="S_SUPPKEY", right_on="L_SUPPKEY", how="inner"
)
N1_S_L = N1_S_L.drop(columns=["S_SUPPKEY", "L_SUPPKEY"])

total2 = N2_C_O.merge(
    N1_S_L, left_on="O_ORDERKEY", right_on="L_ORDERKEY", how="inner"
)
total2 = total2.drop(columns=["O_ORDERKEY", "L_ORDERKEY"])

# concat results
total = pd.concat([total1, total2])

total = total.groupby(["SUPP_NATION", "CUST_NATION", "L_YEAR"], as_index=False).agg(
    REVENUE=pd.NamedAgg(column="VOLUME", aggfunc="sum")
)



In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/q07_rewrite/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 6 ###

total.info()